In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gaurav9712/50-startups/50_Startups.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [3]:
df = np.round(pd.read_csv('/kaggle/input/datasets/gaurav9712/50-startups/50_Startups.csv'))[['R&D Spend','Administration','Marketing Spend','Profit']]
np.random.seed(9)
df

,R&D Spend,Administration,Marketing Spend,Profit
0,165349.0,136898.0,471784.0,192262.0
1,162598.0,151378.0,443899.0,191792.0
2,153442.0,101146.0,407935.0,191050.0
3,144372.0,118672.0,383200.0,182902.0
4,142107.0,91392.0,366168.0,166188.0
5,131877.0,99815.0,362861.0,156991.0
6,134615.0,147199.0,127717.0,156123.0
7,130298.0,145530.0,323877.0,155753.0
8,120543.0,148719.0,311613.0,152212.0
9,123335.0,108679.0,304982.0,149760.0


In [4]:
df = df.iloc[:, 0:-1]
df

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.0,471784.0
1,162598.0,151378.0,443899.0
2,153442.0,101146.0,407935.0
3,144372.0,118672.0,383200.0
4,142107.0,91392.0,366168.0
5,131877.0,99815.0,362861.0
6,134615.0,147199.0,127717.0
7,130298.0,145530.0,323877.0
8,120543.0,148719.0,311613.0
9,123335.0,108679.0,304982.0


In [5]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan
df.head()

/tmp/ipykernel_16/1429726703.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0] = np.nan
/tmp/ipykernel_16/1429726703.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1] = np.nan
/tmp/ipykernel_16/1429726703.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1] = np.nan


,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.0,471784.0
1,NaN,151378.0,443899.0
2,153442.0,101146.0,407935.0
3,144372.0,NaN,383200.0
4,142107.0,91392.0,366168.0


In [6]:
#Step 1 - Impute all missing values with mean of respective columns
df0 = pd.DataFrame()
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [7]:
#0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
0,165349.000000,136898.000000,471784.000000
1,71907.836735,151378.000000,443899.000000
2,153442.000000,101146.000000,407935.000000
3,144372.000000,121399.204082,383200.000000
4,142107.000000,91392.000000,366168.000000
5,131877.000000,99815.000000,362861.000000
6,134615.000000,147199.000000,127717.000000
7,130298.000000,145530.000000,323877.000000
8,120543.000000,148719.000000,311613.000000
9,123335.000000,108679.000000,304982.000000


In [8]:
# Remove thr col 1 imputed value
df1 = df0.copy()
df1.iloc[1,0] = np.nan
df1

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.000000,471784.000000
1,NaN,151378.000000,443899.000000
2,153442.0,101146.000000,407935.000000
3,144372.0,121399.204082,383200.000000
4,142107.0,91392.000000,366168.000000
5,131877.0,99815.000000,362861.000000
6,134615.0,147199.000000,127717.000000
7,130298.0,145530.000000,323877.000000
8,120543.0,148719.000000,311613.000000
9,123335.0,108679.000000,304982.000000


In [9]:
#Use 3 rows to build a model and use the last for prediction
x = df1.iloc[[0,2,3,4],1:3]
x

,Administration,Marketing Spend
0,136898.000000,471784.0
2,101146.000000,407935.0
3,121399.204082,383200.0
4,91392.000000,366168.0


In [10]:
y = df1.iloc[[0,2,3,4],0]
y

0    165349.0
2    153442.0
3    144372.0
4    142107.0
Name: R&D Spend, dtype: float64

In [11]:
lr = LinearRegression()
lr.fit(x, y)
predicted_val = lr.predict(df1.iloc[[1], 1:])
predicted_val

array([156865.90298703])

In [12]:
df1.iloc[3,1] = 23.14

In [13]:
df1

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.00,471784.000000
1,NaN,151378.00,443899.000000
2,153442.0,101146.00,407935.000000
3,144372.0,23.14,383200.000000
4,142107.0,91392.00,366168.000000
5,131877.0,99815.00,362861.000000
6,134615.0,147199.00,127717.000000
7,130298.0,145530.00,323877.000000
8,120543.0,148719.00,311613.000000
9,123335.0,108679.00,304982.000000


In [14]:
#Remove the column 2 imputed value
df1.iloc[3,1] = np.nan
df1

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.0,471784.000000
1,NaN,151378.0,443899.000000
2,153442.0,101146.0,407935.000000
3,144372.0,NaN,383200.000000
4,142107.0,91392.0,366168.000000
5,131877.0,99815.0,362861.000000
6,134615.0,147199.0,127717.000000
7,130298.0,145530.0,323877.000000
8,120543.0,148719.0,311613.000000
9,123335.0,108679.0,304982.000000


In [15]:
#use the last 3 rows to build a model and use the first for the prediction
x = df1.iloc[[0,1,2,4],[0,2]]
x

,R&D Spend,Marketing Spend
0,165349.0,471784.0
1,NaN,443899.0
2,153442.0,407935.0
4,142107.0,366168.0


In [16]:
y = df1.iloc[[0,1,2,4],1]
y

0    136898.0
1    151378.0
2    101146.0
4     91392.0
Name: Administration, dtype: float64

In [17]:
x.iloc[1, 0] = 156865.90298703
lr = LinearRegression()
lr.fit(x, y)
predicted_val = lr.predict(df1.iloc[[3], [0, 2]])
predicted_val

array([114031.94116766])

In [18]:
df1.iloc[3,1] = 11.06

In [19]:
df1

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.00,471784.000000
1,NaN,151378.00,443899.000000
2,153442.0,101146.00,407935.000000
3,144372.0,11.06,383200.000000
4,142107.0,91392.00,366168.000000
5,131877.0,99815.00,362861.000000
6,134615.0,147199.00,127717.000000
7,130298.0,145530.00,323877.000000
8,120543.0,148719.00,311613.000000
9,123335.0,108679.00,304982.000000


In [20]:
#Remove the column 3 imputed value
df1.iloc[4,-1] = np.nan
df1

,R&D Spend,Administration,Marketing Spend
0,165349.0,136898.00,471784.000000
1,NaN,151378.00,443899.000000
2,153442.0,101146.00,407935.000000
3,144372.0,11.06,383200.000000
4,142107.0,91392.00,NaN
5,131877.0,99815.00,362861.000000
6,134615.0,147199.00,127717.000000
7,130298.0,145530.00,323877.000000
8,120543.0,148719.00,311613.000000
9,123335.0,108679.00,304982.000000


In [21]:
#Use last 3 rows to build a model and use the first for the prediction
x = df1.iloc[0:4,0:2]
x

,R&D Spend,Administration
0,165349.0,136898.00
1,NaN,151378.00
2,153442.0,101146.00
3,144372.0,11.06


In [22]:
y = df1.iloc[[0,1,2,4],1]
y

0    136898.0
1    151378.0
2    101146.0
4     91392.0
Name: Administration, dtype: float64